# dataset

> A generic `torch` dataset over the patch layout, hard-example sampling weights, and dataloader construction.

The dataset reads the per-channel patch files, assembles the model input with `channels.build_image`, and returns an `(image, mask)` pair. Binary and multi-class differ only in how the mask is decoded. This module also fixes the original script's missing `build_datasets` helper that `build_dataloaders` relied on.

In [ ]:
#| default_exp dataset

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
import random
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from segmentation_tools_pt.channels import build_image, DEFAULT_CHANNELS
from segmentation_tools_pt.masks import binarize
from segmentation_tools_pt.transforms import get_train_transforms, get_val_transforms

## The dataset

One sample on disk is a set of channel files plus a mask. `__getitem__` loads them, builds the multi-channel image, decodes the mask (binary -> {0,1}, multi-class -> integer labels), and applies the albumentations transform that keeps image and mask aligned.

In [ ]:
#| export
class SegmentationDataset(Dataset):
    """Loads patch samples produced by `dataset_build`.

    Returns `{'image': (C,H,W) float, 'mask': (H,W) long, 'sample_id': str}`.
    """

    def __init__(
        self,
        root: str,                                   # dataset root (has train/ val/ metadata.json)
        split: str = 'train',                        # 'train' or 'val'
        num_classes: int = 1,                        # 1 = binary, >1 = multi-class
        n_source_channels: int = 2,                  # number of raw channel pngs per sample
        channel_names: Sequence[str] = DEFAULT_CHANNELS,  # how to assemble the model image
        transform=None,                              # albumentations Compose or None
    ):
        self.root = Path(root)
        self.split = split
        self.num_classes = num_classes
        self.n_source_channels = n_source_channels
        self.channel_names = list(channel_names)
        self.transform = transform
        self.img_dir = self.root / split / 'images'
        self.mask_dir = self.root / split / 'masks'
        meta_path = self.root / 'metadata.json'
        if meta_path.exists():
            self.metadata = json.loads(meta_path.read_text())[split]
        else:
            ids = (self.root / f'{split}.txt').read_text().split()
            self.metadata = [{'sample_id': s, 'fg_pixels': -1, 'is_empty': False} for s in ids]
        self.sample_ids = [m['sample_id'] for m in self.metadata]

    def __len__(self) -> int:
        return len(self.sample_ids)

    def _load_image(self, sid: str) -> np.ndarray:
        "Read raw channels and assemble the (H, W, C) model input."
        chans = []
        for i in range(self.n_source_channels):
            # Single-channel patches may use the plain patch id; multi-channel use _ch{i}.
            paths = [self.img_dir / f'{sid}_ch{i}.png']
            if i == 0 and self.n_source_channels == 1:
                paths.insert(0, self.img_dir / f'{sid}.png')
            arr = None
            for p in paths:
                arr = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
                if arr is not None:
                    break
            assert arr is not None, f'Missing channel {i} for {sid}'
            chans.append(arr)
        return build_image(chans, self.channel_names)

    def _load_mask(self, sid: str) -> np.ndarray:
        # New patch caches write masks with the same base name; legacy caches use _mask.
        for p in [self.mask_dir / f'{sid}.png', self.mask_dir / f'{sid}_mask.png']:
            m = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
            if m is not None:
                return binarize(m) if self.num_classes == 1 else m.astype(np.int64)
        raise FileNotFoundError(f'Missing mask for {sid}')

    def __getitem__(self, idx: int) -> Dict:
        sid = self.sample_ids[idx]
        image, mask = self._load_image(sid), self._load_mask(sid)
        if self.transform is not None:
            aug = self.transform(image=image, mask=mask)
            image, mask = aug['image'], aug['mask'].long()
        else:
            image = torch.from_numpy(image.transpose(2, 0, 1).astype(np.float32) / 255.0)
            mask = torch.from_numpy(mask.astype(np.int64))
        return {'image': image, 'mask': mask, 'sample_id': sid}

    def get_sample_weights(
        self,
        eps: float = 500.0,      # floor so dense patches are not zero-weighted
        min_fg: int = 500,       # 'boundary' foreground count for empty patches
    ) -> List[float]:
        """Weight hard (sparse-foreground) patches higher for `WeightedRandomSampler`.

        Sparse-foreground patches (partial targets at edges) are harder, so they
        get `1/(fg+eps)`. Empty patches get a neutral boundary weight; unknown
        counts get 1.0.
        """
        boundary = 1.0 / (min_fg + eps)
        out = []
        for m in self.metadata:
            fg = m.get('fg_pixels', -1)
            if fg < 0:
                out.append(1.0)
            elif m.get('is_empty', False):
                out.append(boundary)
            else:
                out.append(1.0 / (fg + eps))
        return out

## Build datasets and loaders

`build_datasets` pairs the train/val splits with their transforms (this is the helper the original script called but never defined). `build_dataloaders` wraps them, oversampling hard patches in training via a `WeightedRandomSampler`.

In [ ]:
#| export
def _seed_worker(worker_id: int) -> None:
    "Seed NumPy/Python RNGs inside each DataLoader worker from torch's worker seed."
    worker_seed = torch.initial_seed() % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def _make_loader_generator(seed: Optional[int]):
    "Return a torch Generator seeded for reproducible sampling, or None."
    if seed is None:
        return None
    generator = torch.Generator()
    generator.manual_seed(int(seed))
    return generator


def build_datasets(
    root: str,                                   # dataset root
    num_classes: int = 1,                        # 1 = binary, >1 = multi-class
    n_source_channels: int = 2,                  # raw channels per sample
    channel_names: Sequence[str] = DEFAULT_CHANNELS,  # model-image assembly
    crop_size: int = 256,                        # crop size for transforms
    mean: Optional[Sequence[float]] = None,      # per-channel mean (defaults to zeros)
    std: Optional[Sequence[float]] = None,       # per-channel std (defaults to ones)
) -> Tuple[SegmentationDataset, SegmentationDataset]:
    """Return `(train_ds, val_ds)` with train/val transforms attached."""
    c = len(channel_names)
    mean = tuple(mean) if mean is not None else (0.,) * c
    std = tuple(std) if std is not None else (1.,) * c
    common = dict(num_classes=num_classes, n_source_channels=n_source_channels, channel_names=channel_names)
    train_ds = SegmentationDataset(root, 'train', transform=get_train_transforms(crop_size, mean, std), **common)
    val_ds = SegmentationDataset(root, 'val', transform=get_val_transforms(crop_size, mean, std), **common)
    return train_ds, val_ds

def build_dataloaders(
    root: str,                  # dataset root
    batch_size: int = 16,       # samples per batch
    num_workers: int = 4,       # dataloader workers (use 2 on network shares)
    crop_size: int = 256,       # crop size
    num_classes: int = 1,       # 1 = binary, >1 = multi-class
    n_source_channels: int = 2, # raw channels per sample
    channel_names: Sequence[str] = DEFAULT_CHANNELS,  # model-image assembly
    mean: Optional[Sequence[float]] = None,  # per-channel mean
    std: Optional[Sequence[float]] = None,   # per-channel std
    pin_memory: bool = True,    # pin host memory for faster H2D copies
    seed: Optional[int] = None, # run seed for reproducible sampling and worker RNGs
) -> Tuple[DataLoader, DataLoader, SegmentationDataset, SegmentationDataset]:
    """Return `(train_loader, val_loader, train_ds, val_ds)`.

    Datasets are returned too so you can inspect samples or re-weight without
    re-instantiating.
    """
    train_ds, val_ds = build_datasets(root, num_classes, n_source_channels, channel_names, crop_size, mean, std)
    sampler = WeightedRandomSampler(
        weights=train_ds.get_sample_weights(),
        num_samples=len(train_ds),
        replacement=True,
        generator=_make_loader_generator(seed),
    )
    worker_init_fn = _seed_worker if num_workers > 0 else None
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        sampler=sampler,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=True,
        persistent_workers=(num_workers > 0),
        worker_init_fn=worker_init_fn,
        generator=_make_loader_generator(seed),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=False,
        persistent_workers=(num_workers > 0),
        worker_init_fn=worker_init_fn,
        generator=_make_loader_generator(None if seed is None else seed + 1),
    )
    return train_loader, val_loader, train_ds, val_ds

## Direct mode (full images, no patch cache)

`DirectDataset` reads from `raw_dir/images/` + `raw_dir/masks/` and applies on-the-fly random/center crop. Use when images are small enough to skip pre-built patches.

In [ ]:
#| export
import albumentations as A

from segmentation_tools_pt.dataset_build import resolve_channel_paths, resolve_mask_path, split_sources
from segmentation_tools_pt.masks import fit_rotated_rects_mask


def split_sample_ids(sample_ids, val_fraction, seed):
    "Shuffle and split sample ids into (train_ids, val_ids)."
    return split_sources(sample_ids, val_fraction, seed)


def _wrap_with_pad(transform, crop_size):
    return A.Compose([
        A.PadIfNeeded(min_height=crop_size, min_width=crop_size, border_mode=cv2.BORDER_REFLECT_101),
        *transform.transforms,
    ])


class DirectDataset(Dataset):
    """Load full-resolution images+masks from raw_dir (no patch cache)."""

    def __init__(
        self, raw_dir, sample_ids, num_classes=1, n_source_channels=1,
        channel_names=DEFAULT_CHANNELS, mask_suffix='_mask', exts='.png',
        fit_rects=False, transform=None,
    ):
        self.raw_dir = Path(raw_dir)
        self.sample_ids = list(sample_ids)
        self.num_classes = num_classes
        self.n_source_channels = n_source_channels
        self.channel_names = list(channel_names)
        self.mask_suffix = mask_suffix
        self.exts = exts
        self.fit_rects = fit_rects
        self.transform = transform

    def __len__(self): return len(self.sample_ids)

    def _load_image(self, sid):
        paths = resolve_channel_paths(self.raw_dir, sid, self.n_source_channels, self.exts)
        chans = [cv2.imread(str(p), cv2.IMREAD_GRAYSCALE) for p in paths]
        for i, c in enumerate(chans):
            assert c is not None, f'Missing channel {i} for {sid}'
        return build_image(chans, self.channel_names)

    def _load_mask(self, sid):
        path = resolve_mask_path(self.raw_dir, sid, self.exts, self.mask_suffix)
        m = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        assert m is not None, f'Missing mask for {sid}'
        if self.fit_rects and self.num_classes == 1:
            m = fit_rotated_rects_mask(m)
        return binarize(m) if self.num_classes == 1 else m.astype(np.int64)

    def __getitem__(self, idx):
        sid = self.sample_ids[idx]
        image, mask = self._load_image(sid), self._load_mask(sid)
        if self.transform is not None:
            aug = self.transform(image=image, mask=mask)
            image, mask = aug['image'], aug['mask'].long()
        else:
            image = torch.from_numpy(image.transpose(2, 0, 1).astype(np.float32) / 255.0)
            mask = torch.from_numpy(mask.astype(np.int64))
        return {'image': image, 'mask': mask, 'sample_id': sid}


def build_direct_datasets(
    raw_dir, train_ids, val_ids, num_classes=1, n_source_channels=1,
    channel_names=DEFAULT_CHANNELS, mask_suffix='_mask', exts='.png',
    fit_rects=False, crop_size=256, mean=None, std=None, aug=None, flips=False,
    boundary_blur=None,
    ):
    c = len(channel_names)
    mean = tuple(mean) if mean is not None else (0.,) * c
    std = tuple(std) if std is not None else (1.,) * c
    common = dict(raw_dir=raw_dir, num_classes=num_classes, n_source_channels=n_source_channels,
                  channel_names=channel_names, mask_suffix=mask_suffix, exts=exts, fit_rects=fit_rects)
    train_tf = _wrap_with_pad(
        get_train_transforms(crop_size, mean, std, aug=aug, flips=flips, boundary_blur=boundary_blur),
        crop_size,
    )
    val_tf = _wrap_with_pad(get_val_transforms(crop_size, mean, std), crop_size)
    return (
        DirectDataset(raw_dir, train_ids, transform=train_tf, **common),
        DirectDataset(raw_dir, val_ids, transform=val_tf, **common),
    )


def build_direct_dataloaders(
    raw_dir, train_ids, val_ids, batch_size=16, num_workers=4, crop_size=256,
    num_classes=1, n_source_channels=1, channel_names=DEFAULT_CHANNELS,
    mask_suffix='_mask', exts='.png', fit_rects=False, mean=None, std=None,
    aug=None, flips=False, boundary_blur=None, pin_memory=True, seed: Optional[int] = None,
    ):
    train_ds, val_ds = build_direct_datasets(
        raw_dir, train_ids, val_ids, num_classes, n_source_channels, channel_names,
        mask_suffix, exts, fit_rects, crop_size, mean, std, aug=aug, flips=flips,
        boundary_blur=boundary_blur,
    )
    worker_init_fn = _seed_worker if num_workers > 0 else None
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=True,
        persistent_workers=(num_workers > 0),
        worker_init_fn=worker_init_fn,
        generator=_make_loader_generator(seed),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=False,
        persistent_workers=(num_workers > 0),
        worker_init_fn=worker_init_fn,
        generator=_make_loader_generator(None if seed is None else seed + 1),
    )
    return train_loader, val_loader, train_ds, val_ds


In [ ]:
# test: end-to-end patch build + dataloader for both multi-channel and single-channel naming
import tempfile
from segmentation_tools_pt.patches import PatchConfig
from segmentation_tools_pt.dataset_build import build_patch_dataset
rng = np.random.default_rng(0)

# --- multi-channel layout (legacy _ch{i} / _mask names) ---
with tempfile.TemporaryDirectory() as d:
    d = Path(d); (d / 'src' / 'images').mkdir(parents=True); (d / 'src' / 'masks').mkdir(parents=True)
    for i in range(4):
        for ch in range(2):
            cv2.imwrite(str(d / 'src' / 'images' / f's{i}_ch{ch}.png'), rng.integers(0, 256, (80, 80), np.uint8))
        m = np.zeros((80, 80), np.uint8); m[20:60, 20:60] = 255
        cv2.imwrite(str(d / 'src' / 'masks' / f's{i}_mask.png'), m)
    cfg = PatchConfig(patch_size=40, stride=40, min_fg_pixels=1, max_empty_fraction=0.0, val_fraction=0.25)
    build_patch_dataset(str(d / 'src'), str(d / 'out'), cfg=cfg, n_channels=2, n_jobs=1)
    tl, vl, tds, vds = build_dataloaders(str(d / 'out'), batch_size=2, num_workers=0, crop_size=32)
    batch = next(iter(tl))
    assert batch['image'].shape == (2, 3, 32, 32)
    assert batch['mask'].shape == (2, 32, 32)
    assert len(tds.get_sample_weights()) == len(tds)

# --- single-channel layout: image/mask patches share the same name ---
with tempfile.TemporaryDirectory() as d:
    d = Path(d); (d / 'src' / 'images').mkdir(parents=True); (d / 'src' / 'masks').mkdir(parents=True)
    for i in range(4):
        cv2.imwrite(str(d / 'src' / 'images' / f's{i}.png'), rng.integers(0, 256, (80, 80), np.uint8))
        m = np.zeros((80, 80), np.uint8); m[20:60, 20:60] = 255
        cv2.imwrite(str(d / 'src' / 'masks' / f's{i}.png'), m)
    cfg = PatchConfig(patch_size=40, stride=40, min_fg_pixels=1, max_empty_fraction=0.0, val_fraction=0.25)
    build_patch_dataset(str(d / 'src'), str(d / 'out'), cfg=cfg, n_channels=1, n_jobs=1)
    tl, vl, tds, vds = build_dataloaders(str(d / 'out'), batch_size=2, num_workers=0, crop_size=32,
                                         n_source_channels=1, channel_names=['ch0'])
    batch = next(iter(tl))
    assert batch['image'].shape == (2, 1, 32, 32)
    assert batch['mask'].shape == (2, 32, 32)
    # filenames in images/ and masks/ should match exactly
    for split in ['train', 'val']:
        assert sorted(p.name for p in (d / 'out' / split / 'images').glob('*.png')) == \
               sorted(p.name for p in (d / 'out' / split / 'masks').glob('*.png'))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()